# Python lab: Volatility Models — ARCH/GARCH

ใช้ Python standard library และกด Run All ตามลำดับ ตัวเลขกับผลตอบแทนแปดค่าเป็นข้อมูลสมมติ ภาพทั้งสามฝังอยู่ใน Notebook แล้ว ไม่มีการดาวน์โหลดข้อมูลตลาด

# Volatility Models — The ARCH Framework

เมื่อวันนี้แกว่งแรง เราควรปรับความเสี่ยงของวันพรุ่งนี้เท่าไร?

ตัวอย่างและกราฟคำนวณจากสมมติฐาน ไม่ใช่ค่าประมาณจากตลาดจริง

บท [Asset Returns — Empirical Stylized Facts](../asset-returns-stylized-facts.html) พาเราเห็นว่า ผลตอบแทนรายวันอาจมี correlation ต่ำ แต่ขนาดผลตอบแทนมักสัมพันธ์กัน วันที่แกว่งแรงจึงเกิดเป็นกลุ่ม คำถามต่อมาคือ เราจะเขียนสิ่งที่เห็นให้เป็นสมการซึ่งอัปเดตความเสี่ยงได้ทุกวันอย่างไร?

กรอบ **ARCH** ให้ความแปรปรวนของผลตอบแทนขึ้นกับข้อมูลที่รู้แล้ว แทนการกำหนด volatility ค่าเดียวให้ทุกวัน หากมีข่าวใหม่หรือผลตอบแทนผิดจากที่คาดมาก แบบจำลองจะปรับความผันผวนของช่วงถัดไป โดยยังเปิดให้ทิศทางของผลตอบแทนเป็นสิ่งที่ทำนายได้ยาก

เราจะเริ่มจากความหมายของ volatility แล้วสร้าง GARCH ทีละส่วน คำนวณตัวอย่างหนึ่งวัน ต่อไปเป็นการพยากรณ์หลายวัน ก่อนเพิ่มความไม่สมมาตรด้วย GJR และดูว่าจะประมาณกับตรวจโมเดลอย่างไร เนื้อหาเรียบเรียงจาก *Volatility Models: the ARCH framework* ของ Stephen Taylor พร้อมตัวอย่างใหม่ที่ตรวจคำนวณแยกจากข้อมูลประวัติศาสตร์ในต้นฉบับ

In [1]:
"""Deterministic ARCH/GARCH teaching examples; Python standard library only.

Returns use decimals (0.01 means 1%). Variances therefore use decimal-return
squared units. The eight observations are handcrafted, not market data.
"""
import math

OMEGA = .000005
ALPHA = .08
BETA = .87
INITIAL_VARIANCE = .0001
EXAMPLE_SHOCK = -.03
EXAMPLE_RETURNS = [.005, -.01, .02, -.03, .01, 0, -.005, .015]


def garch_update(shock, variance, omega=OMEGA, alpha=ALPHA, beta=BETA):
    """Return h_(t+1) after observing epsilon_t and knowing h_t."""
    assert all(math.isfinite(v) for v in (shock, variance, omega, alpha, beta))
    assert variance > 0 and omega > 0 and alpha >= 0 and beta >= 0
    return omega + alpha*shock**2 + beta*variance


def gjr_update(shock, variance, omega=OMEGA, alpha=.03, gamma=.10, beta=BETA):
    """Return GJR h_(t+1), using the indicator epsilon_t < 0."""
    assert all(math.isfinite(v) for v in (shock, variance, omega, alpha, gamma, beta))
    assert variance > 0 and omega > 0 and alpha >= 0 and alpha+gamma >= 0 and beta >= 0
    return omega + (alpha + gamma*(shock < 0))*shock**2 + beta*variance


def long_run_variance(omega, persistence):
    """Finite unconditional variance under unit-variance innovations."""
    assert omega > 0 and 0 <= persistence < 1
    return omega/(1-persistence)


def variance_forecast(first_variance, long_run, persistence, horizon):
    """Return E_t[h_(t+k)], k=1,...,H, for a stationary GARCH recursion.

    The first forecast h_(t+1) is already known at time t. For k>1,
    future squared shocks are replaced by their conditional expectations.
    """
    assert math.isfinite(first_variance) and math.isfinite(long_run)
    assert first_variance > 0 and long_run > 0 and 0 <= persistence < 1
    assert isinstance(horizon, int) and horizon >= 1
    return [long_run + persistence**(k-1)*(first_variance-long_run)
            for k in range(1, horizon+1)]


def persistence_half_life(persistence):
    """Continuous step count for halving a deviation in VARIANCE, not SD."""
    assert 0 < persistence < 1
    return math.log(.5)/math.log(persistence)


def sd_excess_half_life(first_variance, long_run, persistence):
    """Step count to halve sqrt(forecast variance)-sqrt(long-run variance)."""
    assert first_variance > long_run > 0 and 0 < persistence < 1
    target_sd = (math.sqrt(first_variance)+math.sqrt(long_run))/2
    residual_ratio = (target_sd**2-long_run)/(first_variance-long_run)
    return math.log(residual_ratio)/math.log(persistence)


def aggregate_forecast_variance(first_variance, long_run, persistence, horizon):
    """Variance of future residual sum, assuming conditional zero covariances."""
    return sum(variance_forecast(first_variance, long_run, persistence, horizon))


def gaussian_filter(returns, omega=OMEGA, alpha=ALPHA, beta=BETA,
                    mu=0, initial_variance=INITIAL_VARIANCE):
    """Conditional Gaussian log likelihood with a fixed initial variance.

    Each row stores h_t BEFORE seeing r_t. Only after evaluating its density
    is epsilon_t used to produce h_(t+1). The initial h_1 is held fixed across
    the educational parameter grid; it is not estimated from future returns.
    """
    assert math.isfinite(mu) and math.isfinite(initial_variance) and initial_variance > 0
    assert omega > 0 and alpha >= 0 and beta >= 0
    rows, total, variance = [], 0.0, initial_variance
    for time, observed in enumerate(returns, 1):
        assert math.isfinite(observed)
        shock = observed-mu
        contribution = -.5*(math.log(2*math.pi)+math.log(variance)+shock**2/variance)
        rows.append({'time': time, 'return': observed, 'shock': shock,
                     'variance': variance, 'standardized': shock/math.sqrt(variance),
                     'log_likelihood': contribution})
        total += contribution
        variance = garch_update(shock, variance, omega, alpha, beta)
    return {'rows': rows, 'log_likelihood': total, 'next_variance': variance}


def restricted_grid(returns=EXAMPLE_RETURNS, initial_variance=INITIAL_VARIANCE):
    """Compare 15 specified stationary candidates; this is NOT an MLE optimizer."""
    candidates = []
    for omega in [.000003, .000005, .00001]:
        for alpha in [.03, .08, .15]:
            for beta in [.75, .87]:
                if alpha+beta >= 1:
                    continue
                result = gaussian_filter(returns, omega, alpha, beta,
                                         initial_variance=initial_variance)
                candidates.append({'omega': omega, 'alpha': alpha, 'beta': beta,
                                   'log_likelihood': result['log_likelihood']})
    return sorted(candidates, key=lambda row: row['log_likelihood'], reverse=True)


def student_t_unit_variance_scale(degrees_of_freedom):
    """Multiply a conventional t_nu draw by this factor so E[z^2]=1; nu>2."""
    assert degrees_of_freedom > 2
    return math.sqrt((degrees_of_freedom-2)/degrees_of_freedom)


def verify_examples():
    """Hand checks plus chronology and unit-change invariance checks."""
    def close(actual, expected, tol=1e-12):
        assert math.isclose(actual, expected, rel_tol=tol, abs_tol=tol), (actual, expected)

    first = garch_update(EXAMPLE_SHOCK, INITIAL_VARIANCE)
    close(first, .000164)
    close(garch_update(-EXAMPLE_SHOCK, INITIAL_VARIANCE), first)
    close(math.sqrt(first), .012806248474865698)
    long_run = long_run_variance(OMEGA, ALPHA+BETA)
    close(long_run, .0001)
    forecasts = variance_forecast(first, long_run, .95, 5)
    for actual, expected in zip(forecasts, [.000164, .0001608, .00015776, .000154872, .0001521284]):
        close(actual, expected)
    close(aggregate_forecast_variance(first, long_run, .95, 5), .0007895604)
    half_life = persistence_half_life(.95)
    close(.95**half_life, .5)
    close(long_run+.95**half_life*(first-long_run), .000132)
    sd_half_life = sd_excess_half_life(first, long_run, .95)
    sd_after = math.sqrt(long_run+.95**sd_half_life*(first-long_run))
    close(sd_after-math.sqrt(long_run), (math.sqrt(first)-math.sqrt(long_run))/2)
    assert sd_half_life > half_life
    close(gjr_update(.03, INITIAL_VARIANCE), .000119)
    close(gjr_update(-.03, INITIAL_VARIANCE), .000209)
    close(.03+.10/2+.87, .95)  # Requires symmetric, unit-variance innovations.
    filtered = gaussian_filter(EXAMPLE_RETURNS)
    close(filtered['rows'][0]['variance'], INITIAL_VARIANCE)
    close(filtered['rows'][1]['variance'], .000094)
    for index, row in enumerate(filtered['rows']):
        expected = garch_update(row['shock'], row['variance'])
        actual = (filtered['rows'][index+1]['variance'] if index+1 < len(filtered['rows'])
                  else filtered['next_variance'])
        close(actual, expected)
    changed = EXAMPLE_RETURNS[:]
    changed[3] = .06
    perturbed = gaussian_filter(changed)
    for before, after in zip(filtered['rows'][:4], perturbed['rows'][:4]):
        close(before['variance'], after['variance'])
    assert filtered['rows'][4]['variance'] != perturbed['rows'][4]['variance']
    # Changing decimal returns to percentage points scales h and omega by 100^2.
    scale = 100
    scaled = gaussian_filter([scale*r for r in EXAMPLE_RETURNS],
                             omega=OMEGA*scale**2, initial_variance=INITIAL_VARIANCE*scale**2)
    for original, converted in zip(filtered['rows'], scaled['rows']):
        close(converted['variance'], original['variance']*scale**2)
        close(converted['standardized'], original['standardized'])
    close(scaled['log_likelihood'], filtered['log_likelihood']-len(EXAMPLE_RETURNS)*math.log(scale))
    for nu in [3, 5, 10, 30]:
        close(student_t_unit_variance_scale(nu)**2*nu/(nu-2), 1)
    grid = restricted_grid()
    assert len(grid) == 15
    assert all(a['log_likelihood'] >= b['log_likelihood'] for a, b in zip(grid, grid[1:]))
    return {'next_variance': first, 'long_run_variance': long_run,
            'variance_half_life': half_life, 'sd_excess_half_life': sd_half_life,
            'five_day_variance': sum(forecasts),
            'conditional_log_likelihood': filtered['log_likelihood'],
            'grid_candidates': len(grid)}



def close(actual, expected, tol=1e-12):
    assert math.isclose(actual,expected,rel_tol=tol,abs_tol=tol),(actual,expected)
print("Loaded self-contained daily variance calculations; all examples are hypothetical.")

Loaded self-contained daily variance calculations; all examples are hypothetical.


## Volatility คำเดียว ใช้ตอบคนละคำถามได้

[Volatility](../glossary.html#volatility) บอกขนาดการกระจายของผลตอบแทน แต่ก่อนเปรียบเทียบตัวเลขต้องรู้ว่าเป็นค่าอะไร วัดช่วงไหน และใช้ข้อมูลถึงเวลาใด

| ความหมาย | ตัวอย่าง | รู้ได้เมื่อไร |
|---|---|---|
| พารามิเตอร์ของแบบจำลอง | σ ใน GBM ที่สมมติว่าคงที่ | เป็นสมมติฐานหรือค่าที่ประมาณให้โมเดล |
| ค่าที่คำนวณย้อนหลัง | Sample SD ของ returns หรือ √RV จาก intraday returns | หลังมีข้อมูลครบช่วงที่ใช้คำนวณ |
| Conditional volatility | √h ของวันพรุ่งนี้ เมื่อรู้ข้อมูลถึงวันนี้ | เป็นค่าพยากรณ์ตามโมเดลและข้อมูล ณ วันนี้ |
| Stochastic volatility | ความผันผวนแฝงที่มีสมการสุ่มของตัวเอง | ต้องประมาณสถานะแฝงและพารามิเตอร์ |
| Implied volatility | σ ที่ทำให้สูตรราคา Option ให้ราคาตรงกับราคาที่สังเกต | คำนวณย้อนจากราคา Option และสมมติฐานการตั้งราคา |

Sample SD ของผลตอบแทนรายวันหลายวัน กับ [realized variance](../glossary.html#realized-variance) ของวันเดียวจากข้อมูลระหว่างวัน เป็นคนละวิธีวัด แม้จะเรียกกว้าง ๆ ว่า volatility ที่เกิดขึ้นแล้วเหมือนกัน ต้องตรวจความถี่ การหักค่าเฉลี่ย การนับ overnight และหน่วยให้ตรงกัน

ส่วน [implied volatility](../glossary.html#implied-volatility) มีทั้งข้อมูลเกี่ยวกับความเสี่ยงอนาคตและการตั้งราคาความเสี่ยงอยู่ด้วย จึงไม่จำเป็นต้องเท่ากับ GARCH forecast ซึ่งประมาณจาก returns ภายใต้ความน่าจะเป็นของผลลัพธ์จริง

เหตุใด volatility จึงเปลี่ยน? ข่าว ความไม่แน่นอนทางเศรษฐกิจ สภาพคล่อง และพฤติกรรมการซื้อขายล้วนเป็นกลไกที่อาจเกี่ยวข้อง สมการ ARCH จับความสัมพันธ์เชิงเวลาของข้อมูลได้ แต่การประมาณสมการเพียงอย่างเดียวไม่ได้ระบุว่าอะไรเป็นสาเหตุ และ correlation ระหว่างปริมาณซื้อขายกับ volatility ก็ไม่ได้บอกทิศทางเหตุและผล

In [2]:
h_today = INITIAL_VARIANCE
print(f"Daily conditional variance h = {h_today:.6f} decimal-return squared")
print(f"Daily conditional SD = {math.sqrt(h_today):.4%}")
print(f"The same variance in percentage-point squared = {h_today*10000:.4f} %^2")
print("SD and variance are different quantities; h is known before today's return.")

Daily conditional variance h = 0.000100 decimal-return squared
Daily conditional SD = 1.0000%
The same variance in percentage-point squared = 1.0000 %^2
SD and variance are different quantities; h is known before today's return.


## แยกผลตอบแทนเป็นค่าเฉลี่ย ขนาดความเสี่ยง และช็อก

ให้ \(r_t\) เป็น **log return รายวันในหน่วยทศนิยม** เช่น 1% เขียนเป็น 0.01 และให้ \(\mathcal F_{t-1}\) เป็นข้อมูลที่รู้เมื่อสิ้นวัน t−1 เราเขียน

$$
r_t=\mu_t+e_t,\qquad e_t=\sqrt{h_t}\,z_t,
$$

$$
\mu_t=\mathbb E[r_t\mid\mathcal F_{t-1}],\qquad
h_t=\operatorname{Var}(r_t\mid\mathcal F_{t-1}).
$$

\(\mu_t\) คือ conditional mean, \(e_t\) คือผลตอบแทนส่วนที่ผิดจากค่าเฉลี่ยที่คาด และ \(h_t\) คือ [conditional variance](../glossary.html#conditional-variance) ส่วน \(z_t\) คือ standardized innovation ที่สมมติให้มีค่าเฉลี่ยศูนย์และ variance หนึ่ง

ในโมเดลพื้นฐาน เราสมมติ \(z_t\) เป็น iid และเป็นอิสระจากข้อมูลอดีต ถ้าใช้ Normal จะได้

$$
z_t\overset{\mathrm{iid}}{\sim}N(0,1),\qquad
r_t\mid\mathcal F_{t-1}\sim N(\mu_t,h_t).
$$

ก่อนเริ่มวัน t เรารู้ \(\mu_t\) และ \(h_t\) ตามโมเดลแล้ว แต่ยังไม่รู้ \(z_t\) และ \(r_t\) เมื่อวันนั้นจบจึงได้ residual \(e_t\) เพื่อนำไปอัปเดตวันถัดไป การเอา \(r_t\) มาใช้คำนวณ \(h_t\) ของวันเดียวกันในการทดสอบ forecast จะเป็นการใช้ข้อมูลล่วงหน้า

การกำหนด ARCH model จึงต้องระบุให้ครบทั้ง returns และความถี่, ชุดข้อมูลที่ใช้, สมการ mean, สมการ variance, การแจกแจงของ innovation และพารามิเตอร์ทั้งหมด การเปลี่ยนเฉพาะสมการ variance ไม่ได้ตัดสินว่าหางของ innovation เป็น Normal หรือ Student-t

In [3]:
shock = EXAMPLE_SHOCK
standardized_shock = shock/math.sqrt(h_today)
close(standardized_shock, -3)
print(f"Observed shock {shock:.2%}, conditional SD {math.sqrt(h_today):.2%}, z = {standardized_shock:.2f}")
print("One observed z is not a distribution; the model assumes E[z|past]=0 and E[z^2|past]=1.")

Observed shock -3.00%, conditional SD 1.00%, z = -3.00
One observed z is not a distribution; the model assumes E[z|past]=0 and E[z^2|past]=1.


## จาก ARCH ไปเป็น GARCH

ชื่อ **ARCH** ย่อจาก Autoregressive Conditional Heteroskedasticity หมายถึงความแปรปรวนแบบมีเงื่อนไขที่เปลี่ยนไปและอาศัยข้อมูลในอดีต ตัวอย่าง ARCH(1) คือ

$$
h_t=\omega+\alpha e_{t-1}^{2}.
$$

ช็อกเมื่อวานยิ่งมีขนาดใหญ่ ความเสี่ยงวันนี้ยิ่งสูง ถ้าใช้ ARCH หลาย lag ก็รวม \(\alpha_i e_{t-i}^2\) หลายพจน์ วิธีของ **GARCH** เพิ่มความแปรปรวนเดิมเข้าไป ทำให้จดจำอดีตได้ยาวด้วยพารามิเตอร์ไม่มาก

$$
\boxed{h_t=\omega+\alpha e_{t-1}^{2}+\beta h_{t-1}}
$$

ในบทนี้ [GARCH(1,1)](../glossary.html#garch) ใช้ residual ยกกำลังสองย้อนหลังหนึ่ง lag และ variance ย้อนหลังหนึ่ง lag และเริ่มด้วย \(\mu_t=\mu\) คงที่

| พจน์ | หน้าที่ | หน่วย |
|---|---|---|
| ω | ส่วนคงที่ในสมการ variance | ทศนิยม² ต่อช่วงรายวัน |
| αe² | ปรับตามขนาดช็อกที่เพิ่งสังเกต | ทศนิยม² |
| βh | ส่งต่อระดับ variance เดิม | ทศนิยม² |
| α, β | น้ำหนักการตอบสนองและความจำ | ไม่มีหน่วย |

ω ไม่ใช่ long-run variance โดยตัวมันเอง และ α สูงไม่ได้แปลว่า persistence สูงเสมอไป ต้องพิจารณา β ร่วมด้วย

หาก \(0\leq\beta<1\) และพจน์เริ่มต้นหายไปเมื่อขยายกลับไปไกลพอ จะเขียนได้ว่า

$$
h_t=\frac{\omega}{1-\beta}
+\alpha\sum_{j=1}^{\infty}\beta^{j-1}e_{t-j}^{2}.
$$

นี่คือเหตุผลที่สมการสั้น ๆ ยังเก็บอิทธิพลของช็อกหลายวันได้ น้ำหนักของช็อกในประวัติที่สังเกตจริงลดตาม β ส่วนการพยากรณ์อนาคตจะมีอัตราการลดลงอีกตัวคือ α+β เพราะยังต้องเฉลี่ยช็อกที่ยังไม่เกิดด้วย

## ลองอัปเดตจากวันที่ log return เท่ากับ −3%

กำหนดตัวอย่างสมมติให้ \(\mu=0\), \(\omega=0.000005\), \(\alpha=0.08\), \(\beta=0.87\) และ variance ของวันนี้ \(h_t=0.0001\) ดังนั้น SD วันนี้เท่ากับ 1%

เมื่อสิ้นวันพบ \(r_t=-0.03\) จึงได้ \(e_t=-0.03\) แล้วคำนวณสำหรับวันพรุ่งนี้

$$
\begin{aligned}
h_{t+1}
&=0.000005+0.08(-0.03)^2+0.87(0.0001)\\
&=0.000005+0.000072+0.000087\\
&=0.000164.
\end{aligned}
$$

$$
\sigma_{t+1}=\sqrt{0.000164}\approx0.012806=1.2806\%.
$$

log return −3% ในวันนี้ทำให้โมเดลปรับ SD ของวันพรุ่งนี้จาก 1% เป็นประมาณ 1.28% ตัวเลขนี้ไม่ได้บอกว่าจะลงอีก 1.28% และไม่ได้เป็นขอบเขตขาดทุนสูงสุด



Variance 0.000164 ในหน่วยทศนิยม² เท่ากับ 1.64 %² การถอดรากจึงได้ 1.2806% กราฟแสดง variance contributions ไม่ใช่ส่วนประกอบของ SD ที่นำมาบวกกันได้

หาก log return วันนี้เป็น +3% แทน ภายใต้ mean ศูนย์ residual จะเป็น +0.03 แต่ยกกำลังสองได้เท่ากัน GARCH แบบนี้จึงให้ forecast เดิม ความไม่สมมาตรระหว่างข่าวบวกกับข่าวลบต้องใช้สมการเพิ่มเติม

เมื่อแปลง SD รายวันเป็นตัวเลขรายปีด้วยสมมติฐาน 252 วันซื้อขาย

$$
\sigma_{t+1,\mathrm{annualized}}=\sqrt{252h_{t+1}}\approx20.3293\%.
$$

นี่เป็นการรายงานระดับ SD วันพรุ่งนี้บนสเกลรายปี หาก variance ของวันต่อ ๆ ไปเปลี่ยน ค่านี้ไม่ใช่ forecast ของ SD ผลตอบแทนสะสมทั้งปีโดยตรง

In [4]:
contributions = [OMEGA, ALPHA*shock**2, BETA*h_today]
first = garch_update(shock,h_today)
close(sum(contributions),.000164)
close(first,.000164)
close(garch_update(-shock,h_today),first)
print("Variance contributions:", [f"{v:.8f}" for v in contributions])
print(f"Next variance h = {first:.6f}; next daily SD = {math.sqrt(first):.6%}")
print(f"Annualized display convention only, using sqrt(252): {math.sqrt(252*first):.6%}")
print("This display conversion is not a 252-day GARCH forecast.")

Variance contributions: ['0.00000500', '0.00007200', '0.00008700']
Next variance h = 0.000164; next daily SD = 1.280625%
Annualized display convention only, using sqrt(252): 20.329289%
This display conversion is not a 252-day GARCH forecast.


## ระดับระยะยาวและเงื่อนไขที่ทำให้คำนวณได้

สำหรับ GARCH พื้นฐาน เราใช้ \(\omega>0\), \(\alpha,\beta\geq0\) และค่าเริ่มต้น \(h_1>0\) เพื่อให้ variance เป็นบวก กำหนด \(\rho=\alpha+\beta\) ถ้า \(\rho<1\) จะมีคำตอบ stationary ที่มีโมเมนต์อันดับสองจำกัด และ long-run variance เป็น

$$
V=\mathbb E[h_t]=\operatorname{Var}(r_t)
=\frac{\omega}{1-\alpha-\beta}.
$$

หาสูตรได้โดยใช้ \(\mathbb E[e_t^2]=\mathbb E[h_t]=V\) ในสมการเดิม จึงมี \(V=\omega+(\alpha+\beta)V\) สำหรับตัวอย่าง \(\rho=0.95\) และ \(V=0.0001\) หรือ long-run SD เท่ากับ 1% ต่อวัน

คำว่า long-run หมายถึงระดับเฉลี่ยภายใต้พารามิเตอร์และสมมติฐานที่คงเดิม ไม่ได้บอกว่าตลาดต้องกลับมาที่ 1% เมื่อถึงวันใดวันหนึ่ง การเปลี่ยนโครงสร้างตลาดหรือพารามิเตอร์ทำให้ระดับนี้เปลี่ยนได้

**Stationarity ไม่ได้มีความหมายเดียว**

เงื่อนไข α+β&lt;1 ที่ใช้ที่นี่ผูกกับ variance ระยะยาวที่มีค่าจำกัด ส่วน strict stationarity กล่าวถึงการแจกแจงร่วมที่ไม่เปลี่ยนเมื่อเลื่อนเวลา ภายใต้เงื่อนไขมาตรฐานของ GARCH(1,1) เกณฑ์สำคัญของคำตอบ strictly stationary คือ \(\mathbb E[\log(\alpha z_t^2+\beta)]<0\) ซึ่งอาจเป็นจริงได้แม้ unconditional variance ไม่จำกัด

ดังนั้น α+β≥1 ไม่ได้พิสูจน์ว่าไม่มีคำตอบ strictly stationary แต่เราไม่สามารถใช้สูตร V ข้างต้นเป็น variance ระยะยาวที่จำกัดได้ ตัวอย่างและห้องทดลองหลักจึงจำกัด α+β ให้น้อยกว่า 1

In [5]:
rho = ALPHA+BETA
V = long_run_variance(OMEGA,rho)
close(V,.0001)
print(f"Persistence rho = {rho:.4f}; finite unconditional variance = {V:.6f}")
print(f"Long-run daily SD = {math.sqrt(V):.4%}")
fourth_moment_coefficient = 3*ALPHA**2+2*ALPHA*BETA+BETA**2
close(fourth_moment_coefficient,.9153)
print(f"Gaussian GARCH fourth-moment coefficient = {fourth_moment_coefficient:.6f} < 1")
print("alpha+beta<1 gives a finite second moment; finite fourth moments need a stronger condition.")

Persistence rho = 0.9500; finite unconditional variance = 0.000100
Long-run daily SD = 1.0000%
Gaussian GARCH fourth-moment coefficient = 0.915300 < 1
alpha+beta<1 gives a finite second moment; finite fourth moments need a stronger condition.


## Conditional Normal ยังสร้างผลตอบแทนหางหนาได้

การสมมติ \(r_t\mid\mathcal F_{t-1}\) เป็น Normal ไม่ได้บังคับให้ผลตอบแทนที่รวมทุกวันเป็น Normal เพราะแต่ละวันมี variance ต่างกัน เมื่อรวมวันสงบกับวันผันผวน เราได้ variance mixture ที่ต่อจากแนวคิดในบทก่อน

เมื่อ mean คงที่และ variance มีค่าจำกัด residual เป็น martingale difference จึงไม่มี autocorrelation เชิงเส้นข้ามวัน แต่ residuals ยังพึ่งพากันผ่าน variance ได้ ส่วน \(e_t^2\) มักมีความสัมพันธ์ข้ามเวลา นี่ช่วยอธิบาย [volatility clustering](../glossary.html#volatility-clustering)

สำหรับ Normal innovations โมเมนต์อันดับสี่ของ GARCH(1,1) มีค่าจำกัดเมื่อ

$$
3\alpha^2+2\alpha\beta+\beta^2<1.
$$

ในกรณีนั้น kurtosis ของ stationary returns คือ

$$
\kappa_r=
\frac{3[1-(\alpha+\beta)^2]}
{1-(3\alpha^2+2\alpha\beta+\beta^2)}.
$$

ตัวอย่าง α=0.08 และ β=0.87 ให้ \(\kappa_r\approx3.4534\) มากกว่า Normal ที่มี kurtosis 3 หากเงื่อนไขโมเมนต์อันดับสี่ไม่ผ่าน ผลตอบแทนยังอาจมี variance จำกัดแต่ kurtosis ไม่จำกัดได้ ในกรณีนั้น population ACF ของ squared returns ก็ไม่สามารถใช้ตามปกติ เพราะต้องอาศัยโมเมนต์อันดับสี่

ความหนาของหางจึงมีได้ทั้งจาก variance ที่เปลี่ยน และจาก innovation ที่หางหนาแม้ปรับด้วย volatility แล้ว ต้องตรวจสองส่วนนี้แยกกัน หาก α=0 และเริ่มที่ stationary variance โมเดลจะลดเป็น variance คงที่ จึงไม่ได้สร้าง clustering ด้วยตัวมันเอง

In [6]:
# Gaussian innovations can still give an unconditional leptokurtic GARCH mixture.
unconditional_kurtosis = 3*(1-rho**2)/(1-rho**2-2*ALPHA**2)
assert unconditional_kurtosis > 3
print(f"Theoretical unconditional kurtosis under these Gaussian GARCH assumptions = {unconditional_kurtosis:.6f}")
print("This is a model implication when the fourth moment exists, not an empirical estimate.")

Theoretical unconditional kurtosis under these Gaussian GARCH assumptions = 3.453365
This is a model implication when the fourth moment exists, not an empirical estimate.


## พยากรณ์วันถัด ๆ ไปโดยยังไม่รู้ผลตอบแทนของวันเหล่านั้น

เมื่อสิ้นวัน t เราคำนวณ \(h_{t+1}\) ได้จากข้อมูลที่สังเกตแล้ว กำหนด

$$
f_k=\mathbb E[h_{t+k}\mid\mathcal F_t],\qquad f_1=h_{t+1}.
$$

สำหรับ mean คงที่ \(f_k\) เท่ากับ conditional variance ของผลตอบแทนวัน t+k เมื่อมองจากวันนี้ด้วย วันถัดจากนั้นใช้ \(\mathbb E[e_{t+k}^2\mid\mathcal F_t]=f_k\) จึงได้

$$
f_{k+1}=\omega+(\alpha+\beta)f_k,
\qquad
\boxed{f_k=V+\rho^{k-1}(h_{t+1}-V)}.
$$

เราต้องเฉลี่ยช็อกอนาคต ไม่ใช่แทน residual อนาคตทุกวันด้วยศูนย์ การใส่ศูนย์จะเหลืออัตราการลดลง β และได้เส้นพยากรณ์คนละเส้นกับ conditional expectation สูตร recursion นี้สอดคล้องกับวิธี analytical forecasting ใน [เอกสารของแพ็กเกจ arch](https://arch.readthedocs.io/en/stable/univariate/forecasting.html)

กราฟต่อไปตรึง forecast วันแรกไว้ที่ 1.2806% และ long-run SD ที่ 1% แล้วเปรียบเทียบ persistence สามค่า



เส้นแต่ละเส้นแสดง √f ของผลตอบแทนหนึ่งวัน ณ horizon นั้น ไม่ใช่ SD ของผลตอบแทนสะสมถึง horizon และไม่ใช่เส้นทาง volatility ที่รับรองว่าจะเกิดขึ้นจริง

[Half-life](../glossary.html#volatility-persistence) ของ **ส่วนต่าง variance จากระดับระยะยาว** เป็นจำนวนช่วงเพิ่มเติมที่ทำให้ส่วนต่างเหลือครึ่งหนึ่ง

$$
\rho^{H_{1/2}}=\frac12,
\qquad H_{1/2}=\frac{\log(1/2)}{\log\rho},\quad0<\rho<1.
$$

ถ้า ρ=0.95 จะได้ประมาณ 13.51 วันซื้อขาย โดยนับจาก forecast วันแรก จุดที่ variance gap ลดครึ่งหนึ่งมี variance \(V+(h_{t+1}-V)/2=0.000132\) และ SD ประมาณ 1.1489% ซึ่งต่างจากจุดกึ่งกลางของ SD 1% กับ 1.2806% ที่ประมาณ 1.1403% เพราะการถอดรากเป็นฟังก์ชันไม่เชิงเส้น

นอกจากนี้ \(\sqrt{\mathbb E_t[h_{t+k}]}\) ไม่เท่ากับ \(\mathbb E_t[\sqrt{h_{t+k}}]\) โดยทั่วไป กราฟใช้ค่าแรก คือรากของ variance forecast ไม่ได้อ้างว่าเป็น conditional mean ของ volatility แฝงในอนาคต

ลองเพิ่ม persistence โดยคง α กับ V ไว้ในห้องทดลอง สังเกตว่า ω และ β เปลี่ยนตามด้วย จึงเป็นการเปรียบเทียบโมเดลที่ระดับระยะยาวเดียวกัน จากนั้นเปลี่ยนเครื่องหมายช็อก แต่คงขนาดเดิมเพื่อทดสอบความสมมาตร

In [7]:
forecasts = variance_forecast(first,V,rho,60)
close(forecasts[0],first)
close(forecasts[1],.0001608)
print(" k   forecast variance   sqrt(forecast variance), daily")
for k in [1,2,5,10,20,60]:
    print(f"{k:2d}   {forecasts[k-1]:.10f}         {math.sqrt(forecasts[k-1]):.6%}")
variance_half = persistence_half_life(rho)
sd_half = sd_excess_half_life(first,V,rho)
close(rho**variance_half,.5)
assert sd_half > variance_half
print(f"Steps to halve excess variance = {variance_half:.6f}")
print(f"Steps to halve excess square-root forecast SD = {sd_half:.6f}")
print("Steps are measured after k=1. The two half-lives differ because the square root is nonlinear.")
print("sqrt(E[h]) is not E[sqrt(h)]. The former is the conditional SD of the future innovation.")

 k   forecast variance   sqrt(forecast variance), daily
 1   0.0001640000         1.280625%
 2   0.0001608000         1.268069%
 5   0.0001521284         1.233403%
10   0.0001403360         1.184635%
20   0.0001241506         1.114229%
60   0.0001031036         1.015400%
Steps to halve excess variance = 13.513407
Steps to halve excess square-root forecast SD = 14.751340
Steps are measured after k=1. The two half-lives differ because the square root is nonlinear.
sqrt(E[h]) is not E[sqrt(h)]. The former is the conditional SD of the future innovation.


## ความเสี่ยงห้าวันไม่ใช่ความเสี่ยงของวันที่ห้า

ถ้า mean คงที่และ residuals เป็น martingale differences จะมี conditional covariance ข้ามวันที่เป็นศูนย์ ดังนั้นสำหรับผลรวม log returns ใน H วัน

$$
\operatorname{Var}_t\!\left(\sum_{k=1}^{H}r_{t+k}\right)
=\sum_{k=1}^{H}f_k
=HV+(h_{t+1}-V)\frac{1-\rho^H}{1-\rho}.
$$

จากตัวอย่างเดิม forecast variances ห้าวันคือ 0.000164, 0.0001608, 0.00015776, 0.000154872 และ 0.0001521284 ผลรวมเท่ากับ **0.0007895604** จึงได้ SD ของ log return สะสมห้าวันประมาณ **2.8099%**

ถ้าใช้ \(\sqrt{5h_{t+1}}\) จะได้ประมาณ 2.8636% เพราะวิธีนั้นตรึง variance ทุกวันเท่ากับวันแรก ส่วน forecast SD ของเฉพาะวันที่ห้าอยู่ที่ประมาณ 1.2334% ทั้งสามตัวเลขตอบคนละคำถาม

ถ้า mean มีพลวัตที่ทำให้ returns มี covariance ข้ามเวลา ต้องรวมส่วนนั้นก่อน จึงไม่ควรยกสูตรรวม variance นี้ไปใช้ตรง ๆ กับทุก mean model หรือกับ simple returns ที่ทบต้น

Variance forecast ยังไม่เพียงพอสำหรับ [VaR และ ES](../value-at-risk-expected-shortfall.html) ต้องกำหนดการแจกแจงและนิยามขาดทุนด้วย แม้ใช้ Normal innovations รายวัน การแจกแจงผลรวมหลายวันใน GARCH โดยทั่วไปก็ไม่ใช่ Normal พอดี การจำลองหลายวันต้องอัปเดต variance จากช็อกของแต่ละเส้นทาง

In [8]:
H = 5
aggregate = aggregate_forecast_variance(first,V,rho,H)
close(aggregate,.0007895604)
constant_first = H*first
print(f"Sum of {H} forecast variances = {aggregate:.10f}; aggregate SD = {math.sqrt(aggregate):.6%}")
print(f"Holding first-day risk constant: H*h1 = {constant_first:.10f}; SD = {math.sqrt(constant_first):.6%}")
assert aggregate < constant_first
print("Variance addition assumes future residuals have zero conditional pairwise covariances.")
print("A variance forecast alone does not specify a multi-day return quantile or VaR.")

Sum of 5 forecast variances = 0.0007895604; aggregate SD = 2.809912%
Holding first-day risk constant: H*h1 = 0.0008200000; SD = 2.863564%
Variance addition assumes future residuals have zero conditional pairwise covariances.
A variance forecast alone does not specify a multi-day return quantile or VaR.


## GJR: ผลตอบแทนต่ำกว่าที่คาดอาจเพิ่มความเสี่ยงมากกว่า

GARCH แบบเดิมใช้ e² จึงทิ้งเครื่องหมายของ residual แบบจำลอง [GJR-GARCH](../glossary.html#gjr-garch) ของ Glosten, Jagannathan และ Runkle เพิ่ม indicator ที่แยก residual ลบ

$$
h_{t+1}=\omega+\alpha e_t^2+
\gamma\mathbf1_{\{e_t<0\}}e_t^2+\beta h_t.
$$

residual บวกใช้สัมประสิทธิ์ α ส่วน residual ลบใช้ α+γ เมื่อ γ>0 ช็อกลบขนาดเท่ากันจะส่งผลต่อ variance มากกว่า คำว่า “ลบ” หมายถึงต่ำกว่า conditional mean หาก mean ไม่ใช่ศูนย์ ก็ไม่ได้ตรงกับ return ติดลบเสมอไป

เงื่อนไขบวกที่ใช้ได้คือ \(\omega>0\), \(\beta\geq0\), \(\alpha\geq0\) และ \(\alpha+\gamma\geq0\) ถ้า standardized innovations สมมาตรและมี variance หนึ่ง จะมี

$$
\phi=\alpha+\beta+\frac\gamma2,
\qquad V=\frac{\omega}{1-\phi},\quad\phi<1.
$$

ตัวอย่างเปรียบเทียบใช้ GJR ที่ α=0.03, γ=0.10, β=0.87 และ ω=0.000005 ทำให้ φ=0.95 เท่ากับ GARCH เดิม เมื่อ \(h_t=0.0001\)

| Residual ที่สังเกต | GARCH: variance พรุ่งนี้ | GJR: variance พรุ่งนี้ | GJR: SD พรุ่งนี้ |
|---|---:|---:|---:|
| +3% | 0.000164 | 0.000119 | 1.0909% |
| −3% | 0.000164 | 0.000209 | 1.4457% |



News impact curve ตรึง h วันนี้และพารามิเตอร์ แล้วเปลี่ยน residual เพียงตัวเดียว แบบจำลองทั้งสองมี persistence และ long-run variance เท่ากันภายใต้ symmetric innovations แต่แบ่งน้ำหนักช็อกต่างกัน

หาก innovations ไม่สมมาตร ต้องใช้

$$
\phi=\alpha+\beta+
\gamma\mathbb E[z_t^2\mathbf1_{\{z_t<0\}}]
$$

แทนสูตร γ/2 พจน์นี้ขึ้นกับกำลังสองของช็อกฝั่งลบ จึงไม่เท่ากับ probability ของช็อกลบอย่างเดียว

ความไม่สมมาตรมักถูกเรียกว่า leverage effect แต่การพบ γ>0 ยังไม่พิสูจน์ว่าเกิดจากหนี้ต่อทุนเพียงกลไกเดียว สำหรับดัชนี ความสัมพันธ์ระหว่างหุ้นที่เปลี่ยนไปอาจมีส่วนด้วย และแม้ α เป็นศูนย์ ก็หมายเพียงช็อกบวกไม่มีพจน์กระทบโดยตรง ไม่ได้ห้าม variance เพิ่มจากส่วนคงที่และ recursion

In [9]:
for value in [-.03,.03]:
    symmetric = garch_update(value,h_today)
    asymmetric = gjr_update(value,h_today)
    print(f"Shock {value:+.2%}: GARCH h={symmetric:.6f}, GJR h={asymmetric:.6f}")
close(gjr_update(-.03,h_today),.000209)
close(gjr_update(.03,h_today),.000119)
close(.03+.10/2+.87,rho)
print("GJR persistence alpha+beta+gamma/2 assumes symmetric unit-variance innovations.")

Shock -3.00%: GARCH h=0.000164, GJR h=0.000209
Shock +3.00%: GARCH h=0.000164, GJR h=0.000119
GJR persistence alpha+beta+gamma/2 assumes symmetric unit-variance innovations.


## ARCH-in-Mean: ความเสี่ยงช่วยอธิบายค่าเฉลี่ยหรือไม่

กรอบ ARCH เปิดให้ conditional mean เปลี่ยนด้วยได้ ตัวอย่าง ARCH-M ในเอกสารใช้

$$
\mu_t=\mu+\lambda\sqrt{h_t}+\Theta e_{t-1}.
$$

พจน์ λ√h เชื่อมค่าเฉลี่ยกับ conditional SD ส่วน Θe เป็น MA(1) component ที่รับ residual ก่อนหน้า ยังมี specification อีกแบบที่ใช้ λh แต่สองแบบนี้ต่างกันและสัมประสิทธิ์มีหน่วยต่างกัน ต้องอ่านสมการก่อนตีความ

การประมาณ λ บวกไม่ได้แปลว่าผู้ลงทุนจะได้รับผลตอบแทนสูงทุกครั้งที่ volatility สูง ต้องแยกผลตอบแทนคาดหมายจากผลตอบแทนที่เกิดจริง และประเมินความไม่แน่นอนของ λ ด้วย

Taylor ยกตัวอย่างที่ผลทดสอบเปลี่ยนตามการแจกแจงและวิธีหา standard error จึงเหมาะจะอ่านเป็นตัวอย่างว่า “ข้อสรุปไวต่อสมมติฐาน” มากกว่าใช้ยืนยันความสัมพันธ์สากล ห้องทดลองในบทนี้คง mean ไว้ที่ศูนย์ เพื่อให้สูตรพยากรณ์และการรวม variance ยังอยู่ในขอบเขตที่อธิบายไว้

## ประมาณพารามิเตอร์ด้วย likelihood

ก่อนหน้านี้เราเป็นผู้กำหนด ω, α และ β เอง เมื่อนำไปใช้กับข้อมูล เราต้องหา parameter vector θ ที่อธิบายข้อมูลในช่วงประมาณค่าได้ดี โดยกำหนด mean model, variance model และ distribution ให้ครบ

Likelihood มองข้อมูลที่สังเกตเป็นค่าคงที่ แล้วเปลี่ยนพารามิเตอร์เพื่อเปรียบเทียบความหนาแน่นของข้อมูลเหล่านั้น ไม่ใช่ probability ที่ θ เป็นจริง สำหรับ conditional Normal และกำหนดค่าเริ่มต้นไว้ จะมี conditional log-likelihood

$$
\ell(\theta)=\sum_{t=1}^{n}\ell_t(\theta),\qquad
\ell_t=-\frac12\left[
\log(2\pi)+\log h_t+\frac{(r_t-\mu_t)^2}{h_t}
\right].
$$

พจน์ \(e_t^2/h_t\) ลงโทษเมื่อ residual ใหญ่เมื่อเทียบกับความเสี่ยงที่โมเดลให้ ส่วน \(\log h_t\) ป้องกันไม่ให้แก้ปัญหาด้วยการตั้ง variance สูงมากทุกวัน เราหาพารามิเตอร์ที่ทำให้ผลรวมสูงสุด ภายใต้ข้อจำกัดของโมเดล

การคำนวณหนึ่งรอบทำตามเวลาได้ดังนี้

1. เลือก θ และวิธีตั้ง \(h_1\) เช่น variance จากชุดฝึกหรือ backcast ที่ระบุวิธีไว้
2. สำหรับวัน t ใช้ \(\mu_t,h_t\) ที่มีอยู่แล้ว คำนวณ \(e_t,z_t,\ell_t\) จาก \(r_t\)
3. ใช้ residual ของวันนี้อัปเดต mean และ variance ของวันถัดไป
4. รวม \(\ell_t\) ทุกวัน จากนั้น optimizer เปลี่ยน θ แล้วทำซ้ำ

ถ้ากำลังประเมินพยากรณ์ย้อนหลัง ต้องแยกช่วงฝึกกับช่วงทดสอบตามเวลา ค่าเริ่มต้น ค่าเฉลี่ย และพารามิเตอร์ ณ forecast origin ต้องไม่ใช้ข้อมูลอนาคต การใช้ sample variance ของช่วงฝึกทั้งหมดเป็นค่าเริ่มต้นของ likelihood เป็นทางเลือกเชิงประมาณค่า แต่การใช้ทั้งช่วงทดสอบมาตั้งค่าเริ่มต้นจะรั่วข้อมูล

### ตัวอย่างเล็กสำหรับเห็นกลไก

Notebook ใช้ returns ที่กำหนดขึ้นแปดวัน: 0.5%, −1%, 2%, −3%, 1%, 0%, −0.5% และ 1.5% กำหนด mean ศูนย์และ \(h_1=0.0001\) แล้วพิมพ์ตาราง \(r_t,h_t,z_t,\ell_t\) ตามลำดับ ก่อนลองเปรียบเทียบพารามิเตอร์บน grid ที่กำหนด

นี่เป็นการแสดงวิธีคำนวณ likelihood และเลือกจุดที่ดีที่สุด **เฉพาะใน grid** ไม่ใช่ผลประมาณ maximum likelihood ของข้อมูลตลาด ไม่ได้ประมาณ standard errors และจำนวนแปดวันไม่พอสำหรับสรุปคุณภาพโมเดล ในงานประมาณจริงต้องตรวจ convergence, จุดเริ่มต้นหลายชุด, constraints, sensitivity ต่อหน่วยและค่าเริ่มต้น รวมถึงขนาดตัวอย่าง

การคูณ returns ด้วย 100 ต้องคูณ mean ด้วย 100 และ variance รวมถึง ω ด้วย 10,000 ส่วน α และ β คงเดิม ค่า log-likelihood ของข้อมูล n ค่าเลื่อนลง \(n\log100\) ตามการเปลี่ยนหน่วย จึงไม่ควรเทียบค่าดิบข้ามหน่วยหรือข้ามช่วงข้อมูล

In [10]:
result = gaussian_filter(EXAMPLE_RETURNS)
print("Handcrafted returns, decimals:",EXAMPLE_RETURNS)
print(" t   r_t        h_t before r_t     z_t       log-density")
for row in result['rows']:
    print(f"{row['time']:2d}  {row['return']:+.5f}   {row['variance']:.10f}   {row['standardized']:+.5f}   {row['log_likelihood']:.6f}")
print(f"Conditional Gaussian log likelihood = {result['log_likelihood']:.9f}")
close(result['rows'][0]['variance'],.0001)
close(result['rows'][1]['variance'],.000094)

# Perturb r_4: its own prior variance h_4 must not change; h_5 must change.
perturbed_returns = EXAMPLE_RETURNS[:]
perturbed_returns[3] = .06
perturbed = gaussian_filter(perturbed_returns)
for before,after in zip(result['rows'][:4],perturbed['rows'][:4]):
    close(before['variance'],after['variance'])
assert result['rows'][4]['variance'] != perturbed['rows'][4]['variance']
print("No lookahead check passed: changing r_4 affects h_5, not h_4.")

# A density changes numerically when returns are expressed in different units.
scaled = gaussian_filter([100*r for r in EXAMPLE_RETURNS],
                         omega=OMEGA*10000,initial_variance=INITIAL_VARIANCE*10000)
close(scaled['log_likelihood'],result['log_likelihood']-len(EXAMPLE_RETURNS)*math.log(100))
print("Unit-change check passed: log L_percent = log L_decimal - n*log(100).")

candidates = restricted_grid()
assert len(candidates)==15
print("Top 3 of 15 specified stationary candidates; same mu=0 and h1=0.0001:")
for candidate in candidates[:3]:
    print(candidate)
print("Best on this restricted grid is not a continuous-parameter MLE or a fitted market model.")

Handcrafted returns, decimals: [0.005, -0.01, 0.02, -0.03, 0.01, 0, -0.005, 0.015]
 t   r_t        h_t before r_t     z_t       log-density
 1  +0.00500   0.0001000000   +0.50000   3.561232
 2  -0.01000   0.0000940000   -1.03142   3.185254
 3  +0.02000   0.0000947800   +2.05434   1.602888
 4  -0.03000   0.0001194586   -2.74481   -0.169664
 5  +0.01000   0.0001809290   +0.74344   3.113413
 6  +0.00000   0.0001704082   +0.00000   3.419718
 7  -0.00500   0.0001532551   -0.40389   3.391201
 8  +0.01500   0.0001403320   +1.26623   2.715141
Conditional Gaussian log likelihood = 20.819183666
No lookahead check passed: changing r_4 affects h_5, not h_4.
Unit-change check passed: log L_percent = log L_decimal - n*log(100).
Top 3 of 15 specified stationary candidates; same mu=0 and h1=0.0001:
{'omega': 1e-05, 'alpha': 0.08, 'beta': 0.87, 'log_likelihood': 21.191278105125626}
{'omega': 1e-05, 'alpha': 0.15, 'beta': 0.75, 'log_likelihood': 20.973415519974402}
{'omega': 5e-06, 'alpha': 0.08, 'beta'

## หลัง fit แล้ว ยังต้องตรวจสิ่งที่โมเดลเหลือไว้

คำนวณ standardized residuals จากค่าประมาณ

$$
\widehat z_t=\frac{r_t-\widehat\mu_t}{\sqrt{\widehat h_t}}.
$$

ถ้า mean และ variance model ทำหน้าที่ได้ดี เราควรเห็นค่าเฉลี่ยของ z ใกล้ศูนย์ variance ใกล้หนึ่ง และความสัมพันธ์ใน z² ลดลง แต่ sample ที่ดูดีไม่ได้พิสูจน์ว่า z เป็น iid หรือยืนยันว่า forecast อนาคตดี

| สิ่งที่ตรวจ | หากยังพบปัญหา อาจต้องพิจารณา |
|---|---|
| Mean และ ACF ของ z | Mean dynamics หรือข้อมูลที่ยังตกหล่น |
| ACF ของ z² และ residual ARCH tests | Variance dynamics ที่ยังเหลือ |
| Q–Q plot และหางของ z | Innovation distribution ที่ไม่เหมาะ |
| ผลพยากรณ์นอกช่วงฝึก | ความเสถียรของพารามิเตอร์และการเปลี่ยนโครงสร้าง |
| VaR exceptions และการเกิดเป็นกลุ่ม | ทั้งระดับความเสี่ยงและการกระจายที่ใช้หาปลายหาง |

### เปลี่ยน innovation เป็น Student-t

ถ้า \(u\sim t_\nu\) แบบมาตรฐานทั่วไป จะมี variance \(\nu/(\nu-2)\) เมื่อ ν>2 ดังนั้น innovation ที่มี variance หนึ่งต้องใช้

$$
z=\sqrt{\frac{\nu-2}{\nu}}\,u.
$$

ตัวอย่าง ν=8 ต้องคูณ u ด้วย √(6/8) และได้ kurtosis 4.5 เพราะ \(\kappa=3+6/(\nu-4)\) เมื่อ ν>4 ถ้า 2&lt;ν≤4 มี variance จำกัดแต่โมเมนต์อันดับสี่ไม่จำกัด แพ็กเกจสำหรับ ARCH อาจใช้ standardized Student-t อยู่แล้ว จึงต้องตรวจนิยามก่อนปรับสเกลซ้ำ ดู [StudentsT ในเอกสาร arch](https://arch.readthedocs.io/en/stable/univariate/generated/arch.univariate.StudentsT.html)

อีกทางหนึ่งคือใช้ Gaussian objective เป็น [quasi-maximum likelihood (QMLE)](../glossary.html#quasi-maximum-likelihood) แม้ innovation ไม่ใช่ Normal ภายใต้เงื่อนไขด้าน mean, variance, stationarity และโมเมนต์ที่เหมาะสมยังประมาณพารามิเตอร์ได้ แต่ต้องใช้ inference ที่เหมาะ เช่น robust covariance การเปลี่ยนเป็น robust standard errors ไม่ได้ทำให้ Normal tail probabilities ถูกต้องขึ้น และไม่ได้แก้ mean หรือ variance model ที่ระบุผิด

In [11]:
z = [row['standardized'] for row in result['rows']]
print(f"Illustrative residual mean = {sum(z)/len(z):.6f}")
print(f"Illustrative mean z^2 = {sum(value*value for value in z)/len(z):.6f}")
print("Eight handcrafted observations are inadequate for claims about model adequacy.")
print("For an estimated model, inspect standardized residual dependence, squared dependence, tails and out-of-sample forecasts.")

Illustrative residual mean = 0.047986
Illustrative mean z^2 = 1.923411
Eight handcrafted observations are inadequate for claims about model adequacy.
For an estimated model, inspect standardized residual dependence, squared dependence, tails and out-of-sample forecasts.


## การทดสอบช่วยเลือกโมเดล แต่ต้องดูเงื่อนไขด้วย

สำหรับโมเดลซ้อนกัน (nested models) บนข้อมูลและหน่วยเดียวกัน กำหนด \(\ell_0\) กับ \(\ell_1\) เป็น log-likelihood สูงสุดภายใต้ null และ alternative ตามลำดับ สถิติ likelihood ratio คือ

$$
LR=2(\ell_1-\ell_0).
$$

ภายใต้ regularity conditions ที่เหมาะสมและ null อยู่ภายใน parameter space จึงใช้การแจกแจงอ้างอิง \(\chi_d^2\) โดย d เป็นจำนวนข้อจำกัด การเพิ่มพารามิเตอร์หนึ่งตัวอย่างเดียวไม่เพียงพอจะรับรองว่าใช้ χ² หนึ่งองศาอิสระได้ทุกกรณี

ตัวอย่างทดสอบคือ λ=0 ใน ARCH-M เพื่อดูว่ามีหลักฐานของ mean effect หรือไม่ ส่วนการเทียบ Normal กับ Student-t มี Normal เป็นลิมิต ν→∞ หากเขียน ζ=1/ν จะได้ null ζ=0 อยู่บนขอบ parameter space จึงต้องตรวจทฤษฎีของ test หรือใช้ bootstrap ที่ออกแบบให้ตรงกับ null และโมเดล ไม่คัดลอก critical value ของกรณีปกติมาใช้โดยอัตโนมัติ หลักการเรื่อง boundary นี้อธิบายในงานของ [Self และ Liang (1987)](https://doi.org/10.1080/01621459.1987.10478472)

การทดสอบ persistence เท่ากับหนึ่งก็ต้องระวัง asymptotics เช่นกัน และหากใช้ Gaussian QMLE กับ innovation ที่ไม่ใช่ Normal สถิติ likelihood-based แบบปกติอาจต้องปรับให้สอดคล้องกับ misspecification

เมื่อผลทดสอบไม่ถึงเกณฑ์ ให้พูดว่า **ยังไม่ปฏิเสธ null** ไม่ใช่ยอมรับว่าค่าเป็นศูนย์หรือโมเดลเป็นจริง และควรดูขนาด effect, confidence interval กับความสามารถพยากรณ์ร่วมกัน

หากเพิ่ม dummy ของเหตุการณ์ เช่น การเริ่มซื้อขาย Option แล้วทดสอบว่า ω เปลี่ยนหรือไม่ ผลยังเป็นความสัมพันธ์ภายใต้ specification นั้น การเปลี่ยนพร้อมกันของสภาพเศรษฐกิจ สภาพคล่อง และตัวแปรอื่นทำให้ยังสรุปเชิงสาเหตุจาก before/after เพียงอย่างเดียวไม่ได้

## เมื่อขยายไปยังข้อมูลถี่และหลายสินทรัพย์

### Persistence เท่ากับหนึ่งและ EWMA

เมื่อ α+β=1 เราเรียก variance specification ว่า integrated GARCH หรือ IGARCH ภายใต้ ω>0 จะได้ \(f_k=h_{t+1}+(k-1)\omega\) จึงไม่มีการกลับเข้าหา V ที่มีค่าจำกัด ส่วน EWMA แบบ variance recursion เป็นกรณีขอบเขต ω=0 และ α+β=1 ซึ่งให้ forecast variance คงระดับเดิมหากยังไม่มีข้อมูลใหม่ ไม่ได้แปลว่า variance ที่เกิดจริงจะคงที่ทุกวัน

### ข้อมูลระหว่างวัน

การเปลี่ยนจากรายวันเป็นทุกห้านาทีเปลี่ยนทั้งข้อมูลและโจทย์ ต้องคำนึงถึง intraday seasonality, ช่วงเปิด–ปิดตลาด, overnight และ [microstructure noise](../glossary.html#microstructure-noise) ด้วย พารามิเตอร์ GARCH รายวันไม่ควรถูกย้ายไปใช้กับ returns รายห้านาทีโดยไม่ประมาณใหม่ โมเดลหลายองค์ประกอบอาจช่วยแยกส่วนที่ปรับเร็วกับส่วนที่เปลี่ยนช้าได้

### ความเสี่ยงของพอร์ต

สำหรับหลายสินทรัพย์ ให้ \(H_t\) เป็น conditional covariance matrix อาจแยกเป็น

$$
H_t=D_tR_tD_t,\qquad
D_t=\operatorname{diag}(\sqrt{h_{1,t}},\ldots,\sqrt{h_{N,t}}).
$$

D บอก volatility ของแต่ละสินทรัพย์ ส่วน R เป็น conditional correlation matrix ซึ่งต้องมีสมบัติของ correlation matrix ที่ถูกต้อง การ fit GARCH แยกแต่ละสินทรัพย์ยังไม่ทำให้รู้ covariance ทั้งเมทริกซ์ แนวทาง Dynamic Conditional Correlation หรือ DCC เพิ่มโมเดลสำหรับ correlation ที่เปลี่ยนไป

เมื่อ \(w_t\) เป็นน้ำหนักที่รู้ก่อนเริ่มช่วง variance ของผลตอบแทนพอร์ตแบบเชิงเส้นเป็น \(w_t^\top H_tw_t\) สำหรับ simple returns สูตรนี้ใช้ตรงตามน้ำหนักต้นช่วง ส่วนการใช้ log returns รายสินทรัพย์เป็นเพียงการประมาณผลตอบแทนพอร์ต ไม่ใช่ log return ของพอร์ตที่แน่นอน

การประเมินราคาอนุพันธ์ต้องไปอีกขั้นหนึ่ง แบบจำลองที่ fit returns อธิบายความเสี่ยงภายใต้ physical measure การนำ volatility forecast ไปแทน σ ในสูตร Black–Scholes จึงไม่ใช่การสร้างโมเดลราคาแบบ risk-neutral ที่สอดคล้องโดยอัตโนมัติ

In [12]:
for nu in [3,5,10,30]:
    factor = student_t_unit_variance_scale(nu)
    conventional_variance = nu/(nu-2)
    close(factor**2*conventional_variance,1)
    print(f"nu={nu:2d}: Var(T)={conventional_variance:.6f}; z=T*{factor:.6f}; Var(z)=1")
print("For nu<=2, conventional Student t has no finite variance; h cannot retain this unit-variance interpretation.")

nu= 3: Var(T)=3.000000; z=T*0.577350; Var(z)=1
nu= 5: Var(T)=1.666667; z=T*0.774597; Var(z)=1
nu=10: Var(T)=1.250000; z=T*0.894427; Var(z)=1
nu=30: Var(T)=1.071429; z=T*0.966092; Var(z)=1
For nu<=2, conventional Student t has no finite variance; h cannot retain this unit-variance interpretation.


## ทดลองและตรวจคำตอบ

1. ใช้ตัวอย่าง GARCH เดิม แต่เปลี่ยน residual เป็น +3% แล้วเป็น 0% คำนวณ variance และ SD วันพรุ่งนี้ ทั้งสองกรณีบอกอะไรเกี่ยวกับสมมาตรและการปรับกลับ?
2. เมื่อ ρ เพิ่มจาก 0.95 เป็น 0.99 โดยคง V ไว้ที่ 0.0001 และ α=0.08 ค่า ω, β และ variance-gap half-life เปลี่ยนอย่างไร?
3. ทำไม forecast วันที่สองต้องใช้ \(\mathbb E[e_{t+1}^2\mid\mathcal F_t]=h_{t+1}\) แทนการใส่ศูนย์?
4. ตรวจ SD ห้าวันรวมกับ SD ของเฉพาะวันที่ห้าจากตาราง forecast แล้วอธิบายว่าทำไมใช้แทนกันไม่ได้
5. ใน GJR ที่กำหนด ความต่างระหว่าง next-day variance หลัง residual −3% กับ +3% เท่ากับเท่าไร? หากใช้ skewed innovations ยังใช้ γ/2 ได้ทันทีหรือไม่?
6. หลัง fit GARCH แล้ว ACF ของ z² ต่ำลง แต่ Q–Q plot ยังชี้ว่าหางหนา ควรตรวจ mean, variance และ distribution ส่วนใดต่อ? การเปลี่ยน standard errors อย่างเดียวแก้ปัญหาหางได้หรือไม่?

**เปิดแนวคำตอบ**

ข้อ 1 residual +3% ให้ variance 0.000164 และ SD 1.2806% เท่าเดิม ส่วน residual ศูนย์ให้ variance 0.000092 และ SD ประมาณ 0.9592% ช็อกศูนย์เป็นสถานการณ์ที่กำหนด ไม่ใช่ค่าคาดหมายของช็อกยกกำลังสองในอนาคต

ข้อ 2 ω ลดจาก 0.000005 เป็น 0.000001, β เพิ่มจาก 0.87 เป็น 0.91 และ half-life เพิ่มจากประมาณ 13.51 เป็น 68.97 วัน ทั้งหมดเป็นจำนวนช่วงเพิ่มเติมนับจาก variance forecast จุดเริ่มต้น

ข้อ 3 residual มี conditional mean ศูนย์ แต่ conditional second moment ไม่เป็นศูนย์ เพราะยังมีความเสี่ยงอยู่ จึงต้องใช้ variance ของมัน

ข้อ 4 SD ห้าวันรวมประมาณ 2.8099% ขณะที่ SD ของเฉพาะวันที่ห้าประมาณ 1.2334% ค่าแรกวัดการกระจายของผลรวม log returns ส่วนค่าหลังวัดผลตอบแทนวันเดียว

ข้อ 5 ความต่างคือ \(\gamma(0.03)^2=0.00009\) หรือ 0.9 %² ส่วน skewed innovations ต้องคำนวณ \(\mathbb E[z^2\mathbf1_{\{z<0\}}]\) ตาม distribution ที่เลือก

ข้อ 6 ตรวจ innovation distribution และหางเพิ่มเติม พร้อมทดสอบความเสถียรนอกช่วงฝึก ACF ต่ำยังไม่พิสูจน์ iid และ robust standard errors ไม่ได้เปลี่ยน distribution ของผลตอบแทน

[ดาวน์โหลด Python Notebook](volatility-models-arch.ipynb) เพื่อรันตัวอย่าง update, forecast, GJR และ conditional likelihood พร้อมภาพประกอบที่ฝังไว้ ใช้ Python standard library และข้อมูลสมมติที่ระบุครบ ไม่ต้องดาวน์โหลดราคาตลาด

In [13]:
checks = verify_examples()
print("All deterministic assertions passed:")
for name,value in checks.items():
    print(name, value)
print("Try changing the shock magnitude, persistence or grid; keep return and variance units consistent.")

All deterministic assertions passed:
next_variance 0.000164
long_run_variance 9.999999999999992e-05
variance_half_life 13.513407333964874
sd_excess_half_life 14.751340114690096
five_day_variance 0.0007895603999999999
conditional_log_likelihood 20.819183666179942
grid_candidates 15
Try changing the shock magnitude, persistence or grid; keep return and variance units consistent.


## แหล่งที่มาและขอบเขตของบท

แหล่งหลักคือ **Stephen Taylor, *Volatility Models: the ARCH framework*, Certificate in Quantitative Finance, 27 February 2025** จากไฟล์ `JA252.5 Notes.pdf` ที่ผู้ใช้ให้มา เลขหน้าต่อไปนี้นับหน้า PDF เริ่มจากปกเป็นหน้า 1 เพื่อไม่สับสนกับเลขสไลด์และหน้าภาพแทรก

| ส่วนในต้นฉบับ | หน้า PDF | การใช้ในบทนี้ |
|---|---:|---|
| นิยาม volatility และเหตุผลที่เปลี่ยน | 4–7 | ความหมายห้าแบบและข้อจำกัดในการอธิบายสาเหตุ |
| กรอบ ARCH, GARCH และ stylized facts | 8–15, 27–30 | Mean, variance, innovation, recursion และโมเมนต์ |
| Variance forecasts และ half-life | 24–26 | เปลี่ยนดัชนีเวลาให้นับจากสิ้นวัน t และแยก variance จาก SD |
| GJR และ ARCH-M | 31–48 | สมมาตรของ innovations และ SD-in-mean ตามสมการต้นฉบับ |
| Likelihood และการประมาณค่า | 49–55 | คำนวณทีละวันด้วยข้อมูลสมมติและ grid สำหรับสาธิต |
| Hypothesis tests | 56–64 | ขยายเรื่อง null บนขอบ, QMLE และการยังไม่ปฏิเสธ null |
| High-frequency และ multivariate ARCH | 65–67 | เกริ่นข้อจำกัดและแนวทางเรียนต่อ |

ต้นฉบับมีกราฟและค่าประมาณของ DM/USD, GBP/USD, S&P 100 และ FTSE 100 แต่บทนี้ไม่ได้ถอดชุดราคาหรือสร้างกราฟประวัติศาสตร์เหล่านั้นใหม่ ตัวเลขและภาพในบทเป็นตัวอย่างที่กำหนดขึ้น เพื่อให้ตรวจสมการได้โดยไม่อ้างว่าเป็นผล fit ข้อมูลจริง ไม่รวม PDF ต้นฉบับหรือหน้าสไลด์ไว้ในไฟล์เว็บไซต์

สูตรรวม variance หลายวัน, กรณี EWMA, เงื่อนไขโมเมนต์ และตัวอย่างเปรียบเทียบหน่วยเป็นคำอธิบายเสริมที่คำนวณใหม่ การตรวจตัวเลขช่วยยืนยันว่าตัวอย่างสอดคล้องกับสมการ แต่ไม่ได้ตรวจความเหมาะสมของโมเดลกับตลาดใด

อ่านเพิ่มเติมได้จาก [Robert Engle, *GARCH 101* (2001), NYU Faculty Digital Archive](https://archive.nyu.edu/handle/2451/26577) และเอกสารวิธีพยากรณ์ของแพ็กเกจ arch ที่เชื่อมไว้ในเนื้อหา รายละเอียดสมมติฐานและการสร้างภาพเก็บไว้ใน `data/volatility-models-provenance.json`